# Cuaderno 5: poder de la prueba promedio (opción B del protocolo)

**Autor:** Aly Gallo · Continúa los cuadernos 1 a 4.

**Pregunta:** si el protocolo v0.5 pone como prueba **principal** la asociación promedio (¿el BSI basal predice la recuperación en promedio?) y deja el techo como exploratorio, ¿cuánto poder tiene esa prueba con ruido y acoplamiento realistas, y con cuántos pacientes llega a 0,80?

**Cómo:** mismo simulador de los cuadernos 2 a 4 (cohorte real de VitalDB para los tiempos y el BSI basal; recuperación simulada). La prueba es la **pendiente de regresión lineal** (mínimos cuadrados) de la recuperación sobre el BSI basal, unilateral (más BSI, más recuperación). El umbral crítico es el percentil 95 de la pendiente en el mundo **sin efecto**, calculado con la misma simulación, para que absorba el sesgo que producen el ruido y el acoplamiento.

**Mundos con efecto verdadero:** (a) techo moderado y fuerte (los de siempre) y (b) un efecto lineal: la recuperación potencial sube 0,05 o 0,10 por cada SD de BSI.

**Medidas de recuperación:** M0 (misma línea base en predictor y resultado) y M2 (predictor y resultado con extracciones previas independientes). Nada de esto usa desenlaces reales de salud.

In [ ]:
import sys, datetime, json, time, warnings
import numpy as np, pandas as pd
from statsmodels.regression.quantile_regression import QuantReg
warnings.simplefilter("ignore")
print("Fecha:", datetime.date.today(), "| Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

W = 7.0; PRE_DIAS = 7; MIN_POST = 3
SEMILLA = 2026
N_REPS = 100                  # repeticiones por escenario
TECHOS = [0.5, 1.0]           # techo moderado y fuerte
ACOPLAMIENTOS = [0.0, 0.3]
D0, T0, TAU = 1.0, 2.0, 2.0
UMBRAL_CAIDA = 0.5
pruebas = ["crp", "alb", "wbc"]


## Paso 1. Datos y cohorte real (igual que en el cuaderno 2)

In [ ]:
lab = pd.read_csv("https://api.vitaldb.net/labs")
cases = pd.read_csv("https://api.vitaldb.net/cases")
d = lab.merge(cases[["caseid", "subjectid", "opend"]], on="caseid", how="left")
d["dias_post"] = (d.dt - d.opend) / 86400
d["dias_pre"] = -d.dt / 86400

conjuntos = []
for t in pruebas:
    x = d[d.name == t]
    pre = set(x[(x.dt <= 0) & (x.dias_pre <= PRE_DIAS)].caseid)
    p = x[(x.dias_post >= 0) & (x.dias_post <= W)].groupby("caseid").dias_post.agg(["size", "max"])
    post = set(p[(p["size"] >= MIN_POST) & (p["max"] >= W / 2)].index)
    conjuntos.append(pre & post)
coh = set.intersection(*conjuntos)
c = cases[cases.caseid.isin(coh)].sort_values("caseid").drop_duplicates("subjectid", keep="first")
ids = c.caseid.tolist()
print("Cirugias en la cohorte:", len(coh), "| pacientes unicos:", len(ids), "(esperado: 774 y 738)")

base = {}
for t in pruebas:
    x = d[(d.name == t) & d.caseid.isin(ids) & (d.dt <= 0) & (d.dias_pre <= PRE_DIAS)]
    base[t] = x.sort_values("dt").groupby("caseid").result.last()
B = pd.DataFrame(base).loc[ids]
# parametros de estandarizacion (se reutilizan en el Paso A)
def comp(df):
    return pd.concat([-np.log(df.crp + 0.1), df.alb, -np.log(df.wbc)], axis=1)
Z = comp(B)
mu_z, sd_z = Z.mean(), Z.std()
raw = ((Z - mu_z) / sd_z).mean(axis=1)
mu_raw, sd_raw = raw.mean(), raw.std()
bsi_real = ((raw - mu_raw) / sd_raw).values

xc = d[(d.name == "crp") & d.caseid.isin(ids) & (d.dias_post >= 0) & (d.dias_post <= W)]
tiempos = [np.sort(xc[xc.caseid == i].dias_post.values) for i in ids]
m = max(len(t) for t in tiempos)
T = np.full((len(ids), m), np.nan)
for k, t in enumerate(tiempos):
    T[k, :len(t)] = t
N = len(ids)


## Paso B. Simulador con los tres métodos
Cada paciente simulado tiene dos mediciones previas independientes (`base1` y `base2`) del mismo estado verdadero:
- **M0:** predictor `base1`; resultado `(final − nadir)/(base1 − nadir)`.
- **M1:** predictor `base1`; resultado `final` (nivel final, sin cociente).
- **M2:** predictor `base1`; resultado `(final − nadir)/(base2 − nadir)`.
En todos, aquí la prueba principal es la pendiente de mínimos cuadrados de resultado sobre el predictor (la función `delta_beta` del cuaderno 4 sigue disponible, pero no se usa). Se analizan solo pacientes con caída observada > 0,5 SD (mismo criterio para M0 y M1 con `base1`; M2 con `base2`).

In [ ]:
def generar(rng, idx, ruido, acople, techo, gamma=0.0):
    n = len(idx); Tn = T[idx]; mask = ~np.isnan(Tn)
    Bt = bsi_real[idx]
    D = np.clip(D0 * (1 + acople * Bt), 0.3, None)
    u = rng.beta(4, 2, n)
    if gamma:
        u = np.clip(u + gamma * Bt, 0.02, 1.0)
    r = u if techo is None else np.minimum(u, np.clip(0.95 + techo * Bt, 0.05, 1.0))
    k = 1 - np.exp(-(W - T0) / TAU)
    t = np.where(mask, Tn, 0.0)
    rec = r[:, None] * (1 - np.exp(-(t - T0) / TAU)) / k
    q = np.where(t <= T0, t / T0, 1 - rec)
    x = Bt[:, None] - D[:, None] * q
    obs = np.where(mask, x + rng.normal(0, ruido, x.shape), np.nan)
    base1 = Bt + rng.normal(0, ruido, n)
    base2 = Bt + rng.normal(0, ruido, n)
    srt = np.sort(np.where(mask, obs, np.inf), axis=1)
    nadir = srt[:, :2].mean(axis=1)
    last = np.nanargmax(np.where(mask, Tn, -np.inf), axis=1)
    fin = obs[np.arange(n), last]
    return base1, base2, nadir, fin

def resultado(metodo, base1, base2, nadir, fin):
    if metodo == "M0":
        ok = (base1 - nadir) > UMBRAL_CAIDA
        R = np.clip((fin - nadir) / np.where(ok, base1 - nadir, 1), -0.5, 1.5)
    elif metodo == "M1":
        ok = (base1 - nadir) > UMBRAL_CAIDA
        R = fin
    else:  # M2
        ok = (base2 - nadir) > UMBRAL_CAIDA
        R = np.clip((fin - nadir) / np.where(ok, base2 - nadir, 1), -0.5, 1.5)
    return base1[ok], R[ok]

def pend_ols(S, R):
    Sc = S - S.mean()
    return float((Sc * (R - R.mean())).sum() / (Sc ** 2).sum())

def delta_beta(S, R):
    X = np.column_stack([np.ones(len(S)), S])
    b90 = QuantReg(R, X).fit(q=0.90, max_iter=1000).params[1]
    b50 = QuantReg(R, X).fit(q=0.50, max_iter=1000).params[1]
    return b90 - b50


## Paso B. Poder de la prueba promedio
Escenarios: ruido 0,15 y 0,32; acoplamiento c = 0 (sin acoplamiento), 0,15 y 0,32 (el estimado, que probablemente exagera); medidas M0 y M2; n = 150, 738, 2 000, 5 000, 10 000. 150 repeticiones por celda. Tarda unos 15 a 20 minutos.

In [ ]:
rng = np.random.default_rng(SEMILLA + 5)
MUNDOS = [("techo 0.5", dict(techo=0.5)), ("techo 1.0", dict(techo=1.0)),
          ("efecto 0.05/SD", dict(techo=None, gamma=0.05)), ("efecto 0.10/SD", dict(techo=None, gamma=0.10))]
RUIDOS = [0.15, 0.32]; CS = [0.0, 0.15, 0.32]; TAMS = [150, 738, 2000, 5000, 10000]; N_R = 150
filas = []; t_ini = time.time()
for n_ in TAMS:
    for ruido in RUIDOS:
        for cc in CS:
            for met in ["M0", "M2"]:
                nulo = []
                for _ in range(N_R):
                    idx = rng.choice(N, n_, replace=True)
                    S, R = resultado(met, *generar(rng, idx, ruido, cc, None)); nulo.append(pend_ols(S, R))
                crit = np.percentile(nulo, 95)
                fila = {"n": n_, "ruido": ruido, "c": cc, "metodo": met, "pendiente nula, media": round(float(np.mean(nulo)), 3)}
                for nombre, kw in MUNDOS:
                    alt = []
                    for _ in range(N_R):
                        idx = rng.choice(N, n_, replace=True)
                        S, R = resultado(met, *generar(rng, idx, ruido, cc, **kw)); alt.append(pend_ols(S, R))
                    fila["poder " + nombre] = round(float(np.mean(np.array(alt) > crit)), 2)
                filas.append(fila)
    print("n =", n_, "listo;", round(time.time() - t_ini), "s")
tabla = pd.DataFrame(filas)
print(tabla.to_string(index=False))


## Paso C. Tamaño mínimo para poder ≥ 0,80
Para cada escenario (ruido, c, medida, tipo de efecto): el menor n de la rejilla con poder ≥ 0,80, o «> 10 000».

In [ ]:
cols = [c for c in tabla.columns if c.startswith("poder")]
res = []
for (ru, cc, me), g in tabla.groupby(["ruido", "c", "metodo"]):
    fila = {"ruido": ru, "c": cc, "metodo": me}
    for col in cols:
        ok = g[g[col] >= 0.80].n
        fila[col.replace("poder ", "n minimo ")] = int(ok.min()) if len(ok) else "> 10000"
    res.append(fila)
minimos = pd.DataFrame(res)
print(minimos.to_string(index=False))
tabla.to_csv("sim5_poder_prueba_promedio.csv", index=False)
minimos.to_csv("sim5_n_minimo.csv", index=False)
with open("sim5_parametros.json", "w") as f:
    json.dump({"fecha": str(datetime.date.today()), "semilla": SEMILLA + 5, "n_rep": N_R, "ruidos": RUIDOS, "c": CS, "tamanos": TAMS,
               "mundos": [m[0] for m in MUNDOS], "cohorte_n": int(N)}, f, indent=2)
print("Guardado: sim5_poder_prueba_promedio.csv, sim5_n_minimo.csv y sim5_parametros.json")


## Cómo leer los resultados
- **Pendiente nula, media:** debería ser ≈ 0. Si no lo es, el ruido o el acoplamiento sesgan la prueba; el umbral calibrado lo compensa, pero el sesgo es una advertencia.
- **Poder:** se busca ≥ 0,80. Compara con el techo del cuaderno 4 (con n = 10 000 llegaba a 0,53 a 1,00 según el ruido).
- **Paso C:** el n mínimo es lo que habría que pedir en la cohorte final.
- **Límites:** las formas de la recuperación y del efecto son suposiciones; el efecto lineal de 0,05 a 0,10 por SD se eligió para explorar, no es un dato clínico; es una simulación.